# 23 · Silver: ABS population and income by state and year
**Owner: Ar** · Layer: silver · Runs after: 10, 20

**What:** reduce the two ABS files (58,000+ rows across six geography levels) to `silver.abs_state_year`: one row per state per year with population and median income.

| Rule | Problem (profiling evidence) | Treatment |
|---|---|---|
| DQ13 | Notes, a legend and a copyright line sit inside the data | Rows whose `Code` is not a geography code go to `audit.rejects` |
| — | Six geography levels mixed in one file (state, GCCSA, SA4, SA3, SA2, Australia) | Keep the 8 state/territory rows; log the count filtered at each level |
| DQ6 | `-` means "not available" | `-` = NULL here (unlike ICA, where `-` = 0: semantic inconsistency SEM3) |
| DQ14 | Income file repeats five headers | Bronze renamed them `_1`; only the median-income column is used |

In [0]:
%run ./00_common

In [0]:
POP_COL = "estimated_resident_population_no"
INC_COL = "median_total_income_excl_government_pensions_and_allowances"

def geo_level(code):
    c = F.trim(F.col(code))
    return (F.when(c == "AUS", "Australia").when(c.rlike(r"^[1-9]$"), "State/territory")
             .when(c.rlike(r"^\d[A-Z]{4}$"), "Greater capital city").when(c.rlike(r"^\d{3}$"), "SA4")
             .when(c.rlike(r"^\d{5}$"), "SA3").when(c.rlike(r"^\d{9}$"), "SA2").otherwise("not data"))

xw = read("silver", "xw_state")
results = {}
for source, value_col, out_col in [("abs_population", POP_COL, "population"), ("abs_income", INC_COL, "median_income_aud")]:
    b = read("bronze", source).withColumn("geo_level", geo_level("code"))
    rows_in = b.count()
    display(b.groupBy("geo_level").count().orderBy("geo_level"))
    n_rej = reject(b.where("geo_level = 'not data'"), "silver", source, "DQ13", "Footer note, legend, copyright or blank row inside the data")
    states = (b.where(F.trim("code").rlike(r"^[1-8]$"))
               .withColumn("source_value", F.upper(F.trim("label"))).join(xw, "source_value", "inner")
               .select("state_code", F.col("year").cast("int").alias("year"), num(value_col, False).alias(out_col)))
    results[source] = states
    filtered = rows_in - n_rej - states.count()
    log_step("silver", f"filter_{source}", "silver.abs_state_year", rows_in, states.count() + filtered, n_rej,
             f"{filtered:,} rows at other geography levels or 'Other Territories' kept in bronze, not needed in silver")

pop_inc = results["abs_population"].join(results["abs_income"], ["state_code", "year"], "full_outer")
n = write(pop_inc, "silver", "abs_state_year")

In [0]:
display(read("silver", "abs_state_year").groupBy("year")
        .agg(F.count("population").alias("states_with_population"), F.count("median_income_aud").alias("states_with_income"))
        .orderBy("year"))